# 🤖 Caterpillar Predictive Maintenance — Model Training & Evaluation
### CNN-LSTM with Attention | CWRU Bearing Fault Classification

**Models compared:**
- CNN Only (baseline)
- LSTM Only (baseline)
- CNN-LSTM (hybrid)
- CNN-LSTM + Attention (proposed)

**Selection criterion:** Validation Macro F1 (fairest for imbalanced multiclass)

## 📦 0. Imports & Config

In [ ]:
import os, sys, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
import torch.nn as nn
from pathlib import Path
from collections import Counter
from sklearn.metrics import ConfusionMatrixDisplay

warnings.filterwarnings('ignore')
sys.path.append(str(Path('..').resolve()))

from utils.model   import get_model, count_parameters
from utils.trainer import make_loaders, train_model, evaluate, compute_full_metrics, export_onnx, FocalLoss
from utils.data_loader import LABEL_NAMES

# ── Dark theme ────────────────────────────────────────────────────────────────
BG     = '#0f1117'
FG     = '#e0e0e0'
COLORS = ['#4fc3f7', '#66bb6a', '#ffa726', '#ab47bc']
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG,
    'axes.edgecolor': '#444', 'axes.labelcolor': FG,
    'xtick.color': FG, 'ytick.color': FG,
    'text.color': FG, 'grid.color': '#2a2a2a', 'font.size': 11
})

# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR    = Path('../data/processed')
MODEL_DIR   = Path('../models')
MODEL_DIR.mkdir(exist_ok=True)
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE  = 64
EPOCHS      = 50
LR          = 1e-3
PATIENCE    = 10
WINDOW_SIZE = 1024
NUM_CLASSES = 4
CLASS_NAMES = [LABEL_NAMES[k] for k in sorted(LABEL_NAMES)]
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)

print(f'Device  : {DEVICE}')
print(f'Classes : {CLASS_NAMES}')
print(f'Epochs  : {EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {LR}')

---
## 📥 1. Load Processed Data

In [ ]:
X_train = np.load(DATA_DIR / 'X_train.npy')
X_val   = np.load(DATA_DIR / 'X_val.npy')
X_test  = np.load(DATA_DIR / 'X_test.npy')
y_train = np.load(DATA_DIR / 'y_train.npy')
y_val   = np.load(DATA_DIR / 'y_val.npy')
y_test  = np.load(DATA_DIR / 'y_test.npy')

print('Loaded splits:')
print(f'  X_train : {X_train.shape}  y_train: {y_train.shape}')
print(f'  X_val   : {X_val.shape}    y_val  : {y_val.shape}')
print(f'  X_test  : {X_test.shape}   y_test : {y_test.shape}')

# Compute class weights from training labels
counts = np.bincount(y_train)
weights = 1.0 / counts
weights = weights / weights.sum() * NUM_CLASSES
class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
print(f'\nClass weights (for Focal Loss): {weights.round(3)}')

train_loader, val_loader, test_loader = make_loaders(
    X_train, y_train, X_val, y_val, X_test, y_test, BATCH_SIZE
)
print('\n✅ DataLoaders ready')

---
## 🤖 2. Model Architectures
### 2.1 Model Parameter Count

In [ ]:
model_names = ['CNN', 'LSTM', 'CNN-LSTM', 'CNN-LSTM+Attention']

print('=' * 55)
print('MODEL ARCHITECTURE SUMMARY')
print('=' * 55)
for name in model_names:
    m = get_model(name, NUM_CLASSES, WINDOW_SIZE)
    params = count_parameters(m)
    print(f'  {name:<25}: {params:>10,} parameters')
print('=' * 55)

---
## 🏋️ 3. Train All Models

In [ ]:
results = {}

for name in model_names:
    print(f'\n{"="*55}')
    print(f'  Training: {name}')
    print(f'{"="*55}')

    model = get_model(name, NUM_CLASSES, WINDOW_SIZE).to(DEVICE)
    history, train_time, best_val_f1 = train_model(
        model, train_loader, val_loader, DEVICE,
        epochs=EPOCHS, lr=LR, patience=PATIENCE,
        class_weights=class_weights,
    )

    # Evaluate on val set
    criterion = FocalLoss(gamma=2.0, alpha=class_weights)
    _, tr_acc, tr_f1w, tr_f1m, _, _ = evaluate(model, train_loader, criterion, DEVICE)
    _, vl_acc, vl_f1w, vl_f1m, y_pred_val, y_true_val = evaluate(model, val_loader, criterion, DEVICE)

    results[name] = {
        'model'       : model,
        'history'     : history,
        'train_time'  : train_time,
        'train_acc'   : tr_acc,
        'val_acc'     : vl_acc,
        'val_f1_weighted': vl_f1w,
        'val_f1_macro': vl_f1m,
        'train_acc_val': tr_acc,
        'y_pred_val'  : y_pred_val,
        'y_true_val'  : y_true_val,
    }

print('\n✅ All models trained')

---
## 📊 4. Model Comparison
### 4.1 Summary Table

In [ ]:
summary_rows = []
for name, r in results.items():
    summary_rows.append({
        'Model'            : name,
        'Train Acc'        : f"{r['train_acc']*100:.2f}%",
        'Val Acc'          : f"{r['val_acc']*100:.2f}%",
        'Val F1 (weighted)': f"{r['val_f1_weighted']*100:.2f}%",
        'Val F1 (macro)'   : f"{r['val_f1_macro']*100:.2f}%",
        'Train Time (s)'   : f"{r['train_time']:.1f}s",
        'Gap (Acc)'        : f"{(r['train_acc']-r['val_acc'])*100:.2f}%",
    })

summary_df = pd.DataFrame(summary_rows).set_index('Model')
print('\n📊 MULTICLASS MODEL COMPARISON:')
print('=' * 85)
print(summary_df.to_string())
print('=' * 85)

### 4.2 Performance Bar Chart

In [ ]:
metrics = ['train_acc', 'val_acc', 'val_f1_weighted', 'val_f1_macro']
metric_labels = ['Train Acc', 'Val Acc', 'F1 Weighted', 'F1 Macro']
x = np.arange(len(model_names))
width = 0.2

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold', color=FG)

for i, (metric, label) in enumerate(zip(metrics, metric_labels)):
    vals = [results[n][metric] for n in model_names]
    bars = ax.bar(x + i * width, vals, width, label=label, color=COLORS[i], alpha=0.85, edgecolor='#333')

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(model_names, rotation=10)
ax.set_ylabel('Score', color=FG)
ax.set_ylim(0, 1.05)
ax.legend(facecolor='#1e1e2e', edgecolor='#444', labelcolor=FG)
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig('../models/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Overfitting Check — Train vs Val Gap

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Training Curves — Loss & Accuracy per Model', fontsize=14, fontweight='bold', color=FG)

for ax, name, color in zip(axes.flat, model_names, COLORS):
    h = results[name]['history']
    epochs_ran = range(1, len(h['train_loss']) + 1)
    ax2 = ax.twinx()
    ax.plot(epochs_ran, h['train_loss'], color=color, linewidth=1.5, label='Train Loss')
    ax.plot(epochs_ran, h['val_loss'],   color=color, linewidth=1.5, linestyle='--', label='Val Loss')
    ax2.plot(epochs_ran, h['train_acc'], color='#e0e0e0', linewidth=1, alpha=0.6, label='Train Acc')
    ax2.plot(epochs_ran, h['val_acc'],   color='white',   linewidth=1, linestyle=':', label='Val Acc')
    ax.set_title(name, color=color, fontsize=11)
    ax.set_xlabel('Epoch', color=FG)
    ax.set_ylabel('Loss', color=FG)
    ax2.set_ylabel('Accuracy', color=FG)
    ax.grid(alpha=0.15)
    ax.legend(loc='upper right', facecolor='#1e1e2e', edgecolor='#444', labelcolor=FG, fontsize=8)

plt.tight_layout()
plt.savefig('../models/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🏆 5. Best Model Selection

In [ ]:
best_name = max(results, key=lambda n: results[n]['val_f1_macro'])
best_info = results[best_name]
best_model = best_info['model']

print('=' * 60)
print('🏆 BEST MODEL SELECTED')
print('=' * 60)
print(f'  Model              : {best_name}')
print(f'  Selection Criterion: Validation Macro F1')
print(f'  Val Accuracy       : {best_info["val_acc"]*100:.3f}%')
print(f'  Val F1 (weighted)  : {best_info["val_f1_weighted"]*100:.3f}%')
print(f'  Val F1 (macro)     : {best_info["val_f1_macro"]*100:.3f}%')
print(f'  Train-Val Gap      : {(best_info["train_acc"]-best_info["val_acc"])*100:.3f}%')
print(f'  Training Time      : {best_info["train_time"]:.1f}s')
print('=' * 60)

print('\nFull ranking by Val Macro F1:')
ranked = sorted(results.items(), key=lambda x: x[1]['val_f1_macro'], reverse=True)
for rank, (name, r) in enumerate(ranked, 1):
    marker = '🥇' if rank == 1 else ('🥈' if rank == 2 else ('🥉' if rank == 3 else f'  {rank}.'))
    print(f'  {marker} {name:<25} Val F1-macro: {r["val_f1_macro"]*100:.2f}%')

---
## 📊 6. Final Evaluation on Test Set
### 6.1 Predictions on Held-Out Test Set

In [ ]:
criterion = FocalLoss(gamma=2.0, alpha=class_weights)
_, test_acc, test_f1w, test_f1m, y_pred_test, y_true_test = evaluate(
    best_model, test_loader, criterion, DEVICE
)

print(f'Test Set Predictions — {best_name}')
print(f'  Samples   : {len(y_true_test):,}')
print(f'  Accuracy  : {test_acc*100:.3f}%')
print(f'  F1 Macro  : {test_f1m*100:.3f}%')
print(f'  F1 Weighted: {test_f1w*100:.3f}%')

### 6.2 Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true_test, y_pred_test)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Confusion Matrix — {best_name} (Test Set)', fontsize=13, fontweight='bold', color=FG)

# Raw counts
ax = axes[0]
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, colorbar=False, cmap='Blues', values_format=',')
ax.set_title('Raw Counts', color=FG)
ax.tick_params(colors=FG)
ax.xaxis.label.set_color(FG)
ax.yaxis.label.set_color(FG)
ax.set_facecolor(BG)
plt.setp(ax.get_xticklabels(), rotation=20, ha='right')

# Normalized
ax = axes[1]
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=CLASS_NAMES)
disp2.plot(ax=ax, colorbar=False, cmap='Greens', values_format='.2f')
ax.set_title('Normalized (Recall per Class)', color=FG)
ax.tick_params(colors=FG)
ax.xaxis.label.set_color(FG)
ax.yaxis.label.set_color(FG)
ax.set_facecolor(BG)
plt.setp(ax.get_xticklabels(), rotation=20, ha='right')

plt.tight_layout()
plt.savefig('../models/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.3 Full Classification Metrics

In [ ]:
metrics_dict = compute_full_metrics(y_true_test, y_pred_test)

print('\n' + '='*55)
print(f'CLASSIFICATION REPORT — {best_name}')
print('='*55)
print(f'  Accuracy          : {metrics_dict["accuracy"]*100:.3f}%')
print(f'  Precision (macro) : {metrics_dict["precision"]*100:.3f}%')
print(f'  Recall (macro)    : {metrics_dict["recall"]*100:.3f}%')
print(f'  F1 Macro          : {metrics_dict["f1_macro"]*100:.3f}%')
print(f'  F1 Weighted       : {metrics_dict["f1_weighted"]*100:.3f}%')
print(f'  Specificity (avg) : {metrics_dict["specificity"]*100:.3f}%')
print(f'  MCC               : {metrics_dict["mcc"]:.4f}')
print('='*55)

### 6.4 Metrics Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 12), facecolor=BG)
fig.suptitle(f'Classification Dashboard — {best_name}\n(Held-Out Test Set)',
             fontsize=14, fontweight='bold', color=FG)

gauge_metrics = [
    ('Accuracy',    metrics_dict['accuracy'],    COLORS[0]),
    ('Precision',   metrics_dict['precision'],   COLORS[1]),
    ('Recall',      metrics_dict['recall'],      COLORS[2]),
    ('Specificity', metrics_dict['specificity'], COLORS[3]),
    ('F1 Macro',    metrics_dict['f1_macro'],    '#ef5350'),
    ('F1 Weighted', metrics_dict['f1_weighted'], '#26c6da'),
]

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.4)

for idx, (label, value, color) in enumerate(gauge_metrics):
    ax = fig.add_subplot(gs[idx // 3, idx % 3])
    ax.set_facecolor(BG)
    theta = np.linspace(0, np.pi, 200)
    ax.plot(np.cos(theta), np.sin(theta), color='#333', linewidth=8)
    fill_theta = np.linspace(0, np.pi * value, 200)
    ax.plot(np.cos(fill_theta), np.sin(fill_theta), color=color, linewidth=8)
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-0.2, 1.3)
    ax.axis('off')
    ax.text(0, -0.1, f'{value*100:.2f}%', ha='center', va='center',
            fontsize=18, fontweight='bold', color=color)
    ax.text(0, 0.55, label, ha='center', va='center', fontsize=11, color=FG)

# Per-class F1 bar (bottom row)
ax_f1 = fig.add_subplot(gs[2, :])
ax_f1.set_facecolor(BG)
report = metrics_dict['report']
per_class_f1 = [report[str(i)]['f1-score'] for i in range(NUM_CLASSES)]
bars = ax_f1.bar(CLASS_NAMES, per_class_f1, color=COLORS, edgecolor='#333')
for bar, v in zip(bars, per_class_f1):
    ax_f1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
               f'{v*100:.2f}%', ha='center', va='bottom', color=FG, fontsize=10)
ax_f1.set_title('F1 Score per Fault Class', color=FG, fontsize=12)
ax_f1.set_ylabel('F1 Score', color=FG)
ax_f1.set_ylim(0, 1.1)
ax_f1.grid(axis='y', alpha=0.2)

plt.savefig('../models/metrics_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.5 Per-Class Deep Dive

In [ ]:
from sklearn.metrics import classification_report
print('\n' + '='*55)
print(f'PER-CLASS CLASSIFICATION REPORT — {best_name}')
print('='*55)
print(classification_report(y_true_test, y_pred_test, target_names=CLASS_NAMES, digits=4))

### 6.6 Attention Weight Visualization

In [ ]:
if hasattr(best_model, 'get_attention_weights'):
    # Run one batch to get attention weights
    best_model.eval()
    sample_X = torch.tensor(X_test[:8], dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        _ = best_model(sample_X)
    attn = best_model.get_attention_weights()  # (8, T, T)

    fig, axes = plt.subplots(2, 4, figsize=(18, 7))
    fig.suptitle('Attention Weights — CNN-LSTM+Attention (Test Samples)', 
                 fontsize=13, fontweight='bold', color=FG)

    for i, ax in enumerate(axes.flat):
        im = ax.imshow(attn[i].cpu().numpy(), cmap='viridis', aspect='auto')
        ax.set_title(f'Sample {i+1} — {CLASS_NAMES[y_true_test[i]]}', 
                     color=COLORS[y_true_test[i]], fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('../models/attention_weights.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Attention weights not available for this model')

---
## 💾 7. Save Model & Export ONNX

In [ ]:
# Save PyTorch weights
pt_path = MODEL_DIR / f'{best_name.replace("+","_").replace("-","_")}_best.pt'
torch.save(best_model.state_dict(), pt_path)
print(f'✅ PyTorch model saved → {pt_path}')

# Export to ONNX for backend serving
onnx_path = MODEL_DIR / 'fault_classifier.onnx'
export_onnx(best_model, str(onnx_path), WINDOW_SIZE, device=str(DEVICE))

# Verify ONNX inference speed
import onnxruntime as ort
sess = ort.InferenceSession(str(onnx_path))
dummy = np.random.randn(1, 1, WINDOW_SIZE).astype(np.float32)
t0 = time.time()
for _ in range(100):
    sess.run(None, {'vibration': dummy})
avg_ms = (time.time() - t0) / 100 * 1000
print(f'✅ ONNX inference latency: {avg_ms:.2f} ms/sample (target < 20ms)')

---
## 📝 8. Final Summary

In [ ]:
final_summary = {
    'best_model'        : best_name,
    'selection_criterion': 'val_f1_macro',
    'dataset_info'      : {
        'train_samples': int(X_train.shape[0]),
        'val_samples'  : int(X_val.shape[0]),
        'test_samples' : int(X_test.shape[0]),
        'num_classes'  : NUM_CLASSES,
        'class_names'  : CLASS_NAMES,
        'window_size'  : WINDOW_SIZE,
    },
    'model_comparison'  : {
        name: {
            'train_acc'     : float(r['train_acc']),
            'val_acc'       : float(r['val_acc']),
            'val_f1_weighted': float(r['val_f1_weighted']),
            'val_f1_macro'  : float(r['val_f1_macro']),
            'train_time_s'  : float(r['train_time']),
        }
        for name, r in results.items()
    },
    'test_metrics'      : {
        k: float(v) for k, v in metrics_dict.items()
        if k not in ['confusion_matrix', 'report']
    },
    'inference_latency_ms': float(avg_ms),
}

with open(MODEL_DIR / 'final_summary.json', 'w') as f:
    json.dump(final_summary, f, indent=2)

print('=' * 60)
print('  TRAINING PIPELINE SUMMARY')
print('=' * 60)
print(f'  Best Model         : {best_name}')
print(f'  Test Accuracy      : {metrics_dict["accuracy"]*100:.3f}%')
print(f'  Test F1 Macro      : {metrics_dict["f1_macro"]*100:.3f}%')
print(f'  Test F1 Weighted   : {metrics_dict["f1_weighted"]*100:.3f}%')
print(f'  MCC                : {metrics_dict["mcc"]:.4f}')
print(f'  Specificity (avg)  : {metrics_dict["specificity"]*100:.3f}%')
print(f'  ONNX Latency       : {avg_ms:.2f} ms')
print(f'  Summary saved      : models/final_summary.json')
print('=' * 60)
print('\n➡️  Next: Phase 4 — FastAPI Backend')